# Lecture 1 — Pandas in a real workflow

**Hands-on Python for Data Science — PhD Course**

In this lecture we open a real-looking dataset of apartment rentals in Milan
and follow the workflow of a data scientist meeting a new dataset for the first time:

1. Load it and look at it
2. Select and filter
3. Clean it (this is where most of the time goes)
4. Aggregate and summarize
5. Join with other tables
6. Visualize it

Along the way we introduce the parts of `pandas` we need —
**when we need them**, not as an abstract reference manual.

> **A note on LLMs.** You are encouraged to use LLMs (ChatGPT, Claude, Copilot, …)
> alongside this notebook. We will mark a few **🛑 LLM checkpoints** where I
> explicitly suggest you try asking your LLM and then *verify* the answer.
> The skill we are training is not "writing pandas code" — LLMs do that.
> The skill is **knowing what to ask, and recognizing when the answer is wrong**.

---
## 0. Notebooks and Python environments — a quick primer

Before we touch any data, let's make sure we share the same mental model of
the tools we are using. This 10-minute section is for everyone, but especially
for those who tried Jupyter and gave up.

### 0.1 What is a Jupyter notebook?

A notebook is a file made of **cells**. Two main kinds:

- **Markdown cells** — text, like this one.
- **Code cells** — Python code that runs.

The code is executed by a **kernel**, which is a Python process that lives in
the background while the notebook is open. The kernel keeps **state**: every
variable you create stays alive until you restart it.

This has one big consequence that confuses everyone at first:

> ⚠️ **The order in which cells appear is not the order in which they were run.**

You can run cells out of order, edit one, re-run another, and end up in a
state that doesn't match what you see on screen. The number in `[N]` next to
each cell tells you the *execution order*.

**Rule of thumb for sanity:** before you trust a notebook, run *Kernel → Restart
& Run All*. If the notebook breaks, you had hidden state.

### 0.2 What is a Python environment?

A Python *environment* is a folder containing a specific Python interpreter
plus a specific set of installed libraries. Different projects can have
different environments — that's the whole point.

You will hear several names:

| Tool | What it is |
|---|---|
| `venv` | Built-in to Python. Minimal, lightweight. |
| `conda` / `mamba` | Anaconda's environment + package manager. Common in data science. |
| `uv` | Modern, very fast (2024+). Recommended for new projects. |
| `poetry` | Project-oriented, with a `pyproject.toml`. |

If you never created an environment, all your `pip install`s went into your
*system Python*, which is a recipe for trouble. We won't fix that today, but
keep this in mind for the future.

### 0.3 Where do we run code?

Three reasonable choices for this course:

- **Google Colab** — runs in the browser, no install, comes with `pandas`,
  `numpy`, `matplotlib` already there. Easiest.
- **Jupyter locally** — `jupyter lab` or `jupyter notebook` from a terminal.
  Requires a working environment.
- **VS Code** — opens `.ipynb` files natively, good if you also write Python
  scripts and want one tool for everything.

For today, **whatever works for you is fine**. We will not depend on anything
beyond the standard data-science stack.

### 0.4 Reading an error

When something goes wrong, Python prints a **traceback**. The most useful piece
is usually the *last line*: it tells you the type of error and a short message.

Example:
```
KeyError: 'prize'
```
means you asked for a column called `'prize'` and pandas didn't find it.
(Probably you meant `'price'`.)

Don't be afraid of red text. **Read the last line first.**

### 0.5 Importing libraries

Every notebook starts with imports. We will use these throughout the course:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display options for nicer output
pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 120)

# Make plots show up inline (default in Jupyter, explicit doesn't hurt)
%matplotlib inline

The aliases `pd`, `np`, `plt`, `sns` are universal conventions. If you write
`import pandas` and then `pandas.read_csv(...)`, your code works but no human
reader will recognize it as idiomatic. Stick to the conventions.

> **🛑 LLM checkpoint.** Ask your LLM: *"What's the difference between
> `import pandas` and `import pandas as pd`?"* Then ask: *"Why do people
> always write `as pd`?"* The first answer is technical, the second is about
> *convention* — LLMs are good at both, but the second is the kind of thing
> nobody teaches you formally.

---
## 1. Loading the data and the first look

We have a CSV file with rental listings in Milan. Let's open it.

In [ ]:
# Adjust this path to where you saved the file.
# In Colab, after uploading: '/content/rentals_milan.csv'
# Locally: './rentals_milan.csv' or wherever it is.
df = pd.read_csv('rentals_milan.csv')

`pd.read_csv` is one of the most flexible functions in pandas. It has
~50 parameters. We will discover the most useful ones organically.

The first thing to do with a new DataFrame is **look at it from three angles**:

In [ ]:
df.shape

How many rows and columns? Always know this number before doing anything else.

In [ ]:
df.head()

`head()` shows the first 5 rows. Use `df.tail()` for the last 5,
or `df.sample(10)` for 10 random rows (often more informative than the head,
because the head is sometimes systematically different from the rest).

In [ ]:
df.sample(10, random_state=0)

**`random_state=0`** fixes the random seed: the same call gives the same
rows every time. This matters for reproducibility — without it, you cannot
re-run the notebook and get the same output.

### 1.1 What types are the columns?

This is the question that surprises beginners the most.

In [ ]:
df.dtypes

Look carefully. We have:

- `int64` columns (`rooms`, `bedrooms`, `bathrooms`, `deposit_months`)
- `float64` columns (`year_built`, `condo_fees`)
- `bool` for `elevator`
- `object` for **a lot of things** — including `price` and `sqm`!

`object` in pandas usually means *Python strings*. Why are `price` and `sqm`
strings? They should be numbers. Let's investigate.

In [ ]:
df['price'].head(20)

In [ ]:
df['sqm'].head(20)

Some values look like `1500`, others like `'€ 1500'` or `'55 sqm'`.
The presence of even one non-numeric value forces the whole column to be `object`.

> **This is the kind of problem you find every day in real data.** If you don't
> notice it now, every later operation (`.mean()`, `.sum()`, plotting) will
> either fail or — worse — silently do the wrong thing.

We will fix this in the **cleaning** section. For now, just notice it.

### 1.2 A summary in one call

In [ ]:
df.info()

`info()` is one of the most useful methods in pandas. It tells you:

- the index range
- each column's dtype
- the number of **non-null** values per column → from this you can spot missing values
- memory usage

Compare the non-null counts: some columns have fewer than 2513 non-null values.
Those columns have missing data.

In [ ]:
df.describe()

`describe()` shows summary stats for **numeric columns only**.

Notice what is *missing* from this output: `price` and `sqm`. They are not
numeric (they are `object`), so `describe()` ignores them. This is one of the
costs of having dirty types: your summary tools silently skip half the data.

To get a summary of the non-numeric columns:

In [ ]:
df.describe(include='object')

This shows count, unique, top (most frequent), freq.
Already we can see something suspicious: `heating` has `unique = 14`,
even though we'd expect 3 categories (autonomous, central, heat pump).
Why 14? We'll find out shortly.

### 1.3 Missing values at a glance

In [ ]:
df.isna().sum().sort_values(ascending=False)

`isna()` returns a DataFrame of booleans (True where the value is missing).
Summing booleans counts the `True`s.

**Three things to notice:**

1. `energy_class` and `year_built` have many missing values.
2. `nearest_metro` has missing values too — but here it's *expected*: not every
   neighbourhood has a metro stop nearby.
3. The `description` column shows 0 missing — but earlier I said some
   descriptions were empty. **Empty strings are not NaN.** This is a classic
   trap.

In [ ]:
# Empty strings vs NaN: count them separately
(df['description'] == '').sum()

So we have ~250 empty descriptions that `isna()` does **not** count as
missing. Whether to treat empty strings as missing is a *decision* — it
depends on your analysis. The point is: pandas won't make this decision for you,
and neither will an LLM.

> **🛑 LLM checkpoint.** Open your favorite LLM and ask:
> *"How many missing values are in this dataset?"* (paste a few rows or the
> output of `df.isna().sum()`).
>
> See if it mentions the empty-strings issue **without you prompting it**.
> Most won't. This is exactly the kind of domain-aware check you have to add yourself.

---
## 2. Selecting and filtering rows

Real questions about the data sound like:

- *"How many apartments are below 1000 €/month?"*
- *"What are the cheapest rentals in Brera?"*
- *"How many recent buildings (after 2015) have an elevator?"*

To answer them we need to **select** subsets of the DataFrame.

### 2.1 Selecting columns

In [ ]:
# A single column → returns a Series
df['neighborhood'].head()

In [ ]:
# Multiple columns → returns a DataFrame (note the double brackets!)
df[['neighborhood', 'price', 'sqm']].head()

The double brackets are not a typo. `df['neighborhood']` returns a `Series`;
`df[['neighborhood']]` returns a `DataFrame` with one column. Different objects,
different methods available. This is one of the small gotchas that LLMs
sometimes get wrong, mixing the two.

### 2.2 Selecting rows: `.loc` and `.iloc`

There are two access methods, and **they do different things**:

- `.loc[...]` selects by **label** (index value, column name)
- `.iloc[...]` selects by **integer position** (always 0-based)

In [ ]:
# By position
df.iloc[0]          # first row, all columns

In [ ]:
df.iloc[0:3]        # first three rows

In [ ]:
df.iloc[0:3, 1:4]   # first three rows, columns 1–3

In [ ]:
# By label
df.loc[0, 'neighborhood']           # row with index label 0, column 'neighborhood'

In [ ]:
df.loc[0:3, ['neighborhood', 'price']]   # rows 0–3 (inclusive!), specific columns

⚠️ **Watch this difference**: `iloc[0:3]` returns 3 rows (0, 1, 2 — Python slice).
`loc[0:3]` returns 4 rows (0, 1, 2, 3 — both endpoints inclusive). This is
genuinely confusing the first time. The reason is that `.loc` works on labels,
not positions, and slicing labels inclusively is more intuitive when labels
are non-numeric (like dates).

### 2.3 Boolean filtering — the workhorse

This is how you actually answer questions on data.

In [ ]:
# Apartments under 1000 €/month — but wait...
try:
    result = df[df['price'] < 1000]
    print(f"Returned {len(result)} rows.")
except TypeError as e:
    print(f"Got a TypeError: {e}")

Boom. The `price` column is `object` (it contains strings like `'€ 1500'`),
so the comparison `< 1000` is asking pandas to compare strings to integers,
which it refuses to do.

This is actually the *good* outcome — we get a loud error. In older pandas
versions, or in some cases, the comparison silently returns wrong results.
Either way, **the lesson is the same**: never trust comparison/aggregation on
columns whose dtype you haven't checked.

We'll fix this in the cleaning section. For now, let's filter on a column
that *is* clean:

In [ ]:
# Apartments with elevator, on a high floor
df[(df['elevator'] == True) & (df['year_built'] > 2015)].head()

Two things to notice:

1. **Parentheses are required** around each condition when combining with `&` (and) or `|` (or).
   Forgetting them gives a confusing error about ambiguous truth values.
2. We use `&`/`|`, **not** `and`/`or`. The Python keywords `and`/`or` work on
   single booleans, not on Series.

In [ ]:
# .isin() for "value in list"
df[df['neighborhood'].isin(['Brera', 'Quadrilatero', 'Centro Storico'])].head()

> **🛑 LLM checkpoint.** Ask your LLM: *"Find all apartments in Brera
> under 2000 €/month."* Look carefully at the answer.
>
> Did it write `df[df['price'] < 2000]` without checking the dtype? Did it
> notice that `'Brera'` might also be spelled `'brera'` or `'BRERA'` in the data?
> If neither — that's why we are still in the room.

---
## 3. Data cleaning — where the real work happens

Cleaning is typically 60–80% of the time on any real dataset.
This section is the longest of the lecture for that reason.

We'll fix, in order:

1. Mixed types (`price`, `sqm`, `floor`)
2. Inconsistent encoding (`heating`, `furnishing`, `neighborhood`)
3. Date formats (`listing_date`)
4. Missing values (and the difference between *missing at random* and *structurally missing*)
5. Duplicates
6. Logical errors (negative deposits, impossible rooms, etc.)

### 3.1 Mixed types in `price` and `sqm`

Let's see exactly which values are problematic.

In [ ]:
# Find rows where prezzo is not purely a number
non_numeric_prezzo = df['price'].astype(str).str.contains(r'[^\d]', regex=True)
df.loc[non_numeric_prezzo, 'price'].head(10)

Some have a `€` sign. To convert them all to numbers, we need to:

1. Make sure everything is a string
2. Strip non-digit characters
3. Convert to numeric

A common mistake is to call `pd.to_numeric` directly:

In [ ]:
# This will FAIL or coerce to NaN — try it
pd.to_numeric(df['price'], errors='coerce').isna().sum()

`errors='coerce'` turns failed conversions into `NaN`. We just lost
~125 rows of price information silently. **Always check how many NaNs
appeared after a conversion.**

The right approach: clean *first*, convert *after*.

In [ ]:
# Strip currency symbols, spaces, and 'sqm' units
df['price'] = (df['price'].astype(str)
                .str.replace('€', '', regex=False)
                .str.replace(' ', '', regex=False)
                .astype(int))

df['sqm'] = (df['sqm'].astype(str)
            .str.replace('sqm', '', regex=False)
            .str.strip()
            .astype(int))

df[['price', 'sqm']].dtypes

Now `price` and `sqm` are integers. Let's verify nothing was lost:

In [ ]:
df.shape  # should still be 2513

In [ ]:
df[['price', 'sqm']].describe()

### 3.2 The `floor` column — a real headache

This column has values like `'T'`, `'PT'`, `'piano terra'`, `'1'`, `'1°'`, `'primo'`, `'5'`...

In [ ]:
df['floor'].value_counts().head(20)

A naive approach would be `pd.to_numeric(errors='coerce')` — but that
throws away every textual value. We'd lose all ground floors. Instead, we
build an **explicit mapping**.

In [ ]:
# Define a mapping function
def normalize_floor(x):
    x = str(x).strip().lower()
    # Ground floor variants
    if x in ['g', 'gf', 'ground floor', 'ground', '0']:
        return 0
    if x in ['1', '1st', 'first']:
        return 1
    # Try direct conversion (for plain numbers like '5' or '5th')
    try:
        return int(x.replace('th', '').replace('rd', '').replace('nd', '').replace('st', ''))
    except ValueError:
        return None  # we'll see what's left

df['floor'] = df['floor'].apply(normalize_floor)
df['floor'].value_counts(dropna=False).sort_index()

**`apply` applied a function to every value in the column.** It's the
swiss-army knife of pandas: when no built-in method does what you want, write
a function and apply it.

Now `floor` is numeric (with possibly some `None`/NaN if some values weren't
covered by our mapping). Always check for what got lost:

In [ ]:
df['floor'].isna().sum()

### 3.3 Inconsistent categorical values

Look at `heating`:

In [ ]:
df['heating'].value_counts()

We have 14 distinct values where there should be 3. Most are
trivial variants: spaces, capitalization, abbreviations.

A naive cleanup is `.str.strip().str.lower()`:

In [ ]:
df['heating'].str.strip().str.lower().value_counts()

That helped — we went from 14 to fewer values — but **abbreviations
are still there** (`'auto.'`, `'centr.'`, `'pdc'`). This is exactly where LLM
suggestions often stop, and where a human eye is needed.

We need a proper mapping:

In [ ]:
heating_map = {
    'autonomous': 'autonomous',
    'auto.': 'autonomous',
    'central': 'central',
    'centr.': 'central',
    'heat pump': 'heat pump',
    'heatpump': 'heat pump',
    'hp': 'heat pump',
}

df['heating'] = (df['heating']
                       .str.strip().str.lower()
                       .map(heating_map))

df['heating'].value_counts(dropna=False)

**`map`** is similar to `apply`, but takes a *dictionary* (or function)
and is typically used for value-by-value substitutions. The result is now
clean: 3 categories, no NaN.

> **Pattern to remember.** Whenever you "clean" a categorical column, the
> diagnostic is simple: **count the categories before and after, and check
> that the after count matches your expectation.** If you expected 3 and got
> 5, you didn't finish the job.

Apply the same logic to `furnishing` and `neighborhood` (left as exercise — or as
LLM checkpoint).

> **🛑 LLM checkpoint.** Ask your LLM: *"Clean the `furnishing` column
> in this DataFrame: it should have only 3 distinct values."*
>
> Then check: did it produce a mapping like ours, or did it just call
> `.str.lower().str.strip()` and stop? If it stopped, count the unique values
> in the result. Are they 3?

In [ ]:
# Quick clean of `furnishing`
furnishing_map = {
    'furnished': 'furnished',
    'furn.': 'furnished',
    'unfurnished': 'unfurnished',
    'unfurn.': 'unfurnished',
    'empty': 'unfurnished',
    'partly furnished': 'partly furnished',
    'partly furn.': 'partly furnished',
    'partly': 'partly furnished',
}

df['furnishing'] = (df['furnishing']
                     .str.strip().str.lower()
                     .map(furnishing_map))

df['furnishing'].value_counts(dropna=False)

In [ ]:
# Clean `neighborhood`: strip 'Zone ' prefix, normalize case
df['neighborhood'] = (df['neighborhood']
              .str.replace(r'^Zone\s+', '', regex=True, case=False)
              .str.strip()
              .str.title())

df['neighborhood'].value_counts().head(10)

### 3.4 Date formats

`listing_date` has dates in multiple formats: ISO `2025-03-15`, European
`15/03/2025`, US `03/15/2025` (**ambiguous!**), and even `'15 March 2025'`.

In [ ]:
df['listing_date'].head(15)

The naive call:

In [ ]:
# Try the naive approach
parsed_naive = pd.to_datetime(df['listing_date'], errors='coerce')
parsed_naive.isna().sum()

Some are NaT (Not a Time — pandas' missing-date marker). Worse:
the ones that *parsed* may have day and month swapped due to the US format.
This is the most insidious kind of bug, because it doesn't raise an error.

A more careful approach: try multiple formats explicitly.

In [ ]:
def parse_date(s):
    if pd.isna(s):
        return pd.NaT
    s = str(s).strip()
    formats_to_try = ['%Y-%m-%d', '%d/%m/%Y', '%m/%d/%Y']
    for fmt in formats_to_try:
        try:
            return pd.to_datetime(s, format=fmt)
        except (ValueError, TypeError):
            continue
    # Extended format: "15 March 2025"
    months = {
        'january': '01', 'february': '02', 'march': '03', 'april': '04',
        'may': '05', 'june': '06', 'july': '07', 'august': '08',
        'september': '09', 'october': '10', 'november': '11', 'december': '12'
    }
    for month_name, month_num in months.items():
        if month_name in s.lower():
            parts = s.lower().replace(month_name, month_num).split()
            if len(parts) == 3:
                try:
                    return pd.to_datetime(f"{parts[0]}-{parts[1]}-{parts[2]}",
                                          format='%d-%m-%Y')
                except ValueError:
                    pass
    return pd.NaT

df['listing_date'] = df['listing_date'].apply(parse_date)
df['listing_date'].dtype

**Even this is imperfect.** When we encounter `'03/05/2025'`, we still
don't know if it's March 5 or May 3. Real data ingestion sometimes requires
a domain choice (e.g. "we know the source is Italian, so dd/mm always wins").
The lesson is: *parsing dates is not a one-liner*.

In [ ]:
df['listing_date'].describe()

### 3.5 Missing values — and a crucial concept

A common reflex is to call `df.dropna()` or `df.fillna(0)` and move on.
**Don't.** Missing values can mean very different things:

- **Missing completely at random (MCAR)** — the missingness has no pattern.
  Rare in practice.
- **Missing at random (MAR)** — missingness depends on observed variables.
- **Missing not at random (MNAR)** — missingness depends on the missing value itself.

These distinctions affect what's safe to do.

Look at our `energy_class` column:

In [ ]:
df['energy_class'].isna().sum()

In [ ]:
# Where are the missing values? Plot the distribution of construction year for missing vs not
df.groupby(df['energy_class'].isna())['year_built'].agg(['mean', 'count'])

See how the average construction year for the rows with missing energy
class is much **lower** than for the others. **The missingness is not random.**

This makes sense: the *Attestato di Prestazione Energetica* (APE) became
mandatory in Italy only in 2005. Buildings constructed before that often
don't have a recorded energy class.

What does this mean for our analysis?

- `dropna()` would systematically delete old buildings → biased dataset.
- `fillna('C')` (the most common class) would say old buildings have average
  efficiency, which is the opposite of true.
- A reasonable choice is to **keep NaN as a category of its own** ("unknown"),
  because the fact of being missing carries information.

In [ ]:
# Treat missing as its own category
df['energy_class'] = df['energy_class'].fillna('Sconosciuta')
df['energy_class'].value_counts()

> **🛑 LLM checkpoint — important.** Paste a few rows of this dataset to
> your LLM and ask: *"How should I handle the missing values in
> `energy_class`?"*
>
> The most likely answers are: drop, fill with mode, or fill with a constant.
> Most LLMs **will not investigate whether the missingness is structured**
> unless you explicitly tell them to. This is a research-quality decision,
> and the responsibility for it is yours.

### 3.6 Duplicates

In [ ]:
# Exact duplicates (same content, possibly different `id`)
exact_dupes = df.duplicated(subset=df.columns.difference(['id']), keep=False)
exact_dupes.sum()

We pass `subset=...` to ignore `id` (which is unique by construction).
`keep=False` marks **all** duplicate rows (not just the second occurrence),
which is what we want for inspection.

Let's drop them, keeping the first occurrence:

In [ ]:
before = len(df)
df = df.drop_duplicates(subset=df.columns.difference(['id']), keep='first')
print(f"Removed {before - len(df)} exact duplicates. Now: {len(df)} rows.")

**Always print "before / after" counts.** If you don't, you have no
guarantee that the operation did what you expected.

Near-duplicates (same listing reposted with a slightly different price) are
harder. There's no built-in for them — you'd need a custom rule, e.g.
"same `sqm`, same `neighborhood`, same `year_built`, prices within 10% of each
other". We won't tackle that today, but **be aware** that `drop_duplicates`
is not the end of the story.

### 3.7 Logical errors

Some values are simply impossible. We can find them with sanity checks.

In [ ]:
# Negative deposits?
df[df['deposit_months'] < 0]

In [ ]:
# Zero square meters?
df[df['sqm'] == 0]

In [ ]:
# More rooms than total rooms?
df[df['bedrooms'] > df['rooms']]

In [ ]:
# Construction year outside [1800, current year]?
df[(df['year_built'] < 1800) | (df['year_built'] > 2025)]

These are all clearly errors. Decide case by case whether to drop the
row, replace the value with NaN, or correct it manually. For this lecture,
let's drop them:

In [ ]:
mask_valid = (
    (df['deposit_months'].fillna(0) >= 0) &
    (df['sqm'] > 0) &
    (df['bedrooms'] <= df['rooms']) &
    (df['year_built'].fillna(2000).between(1800, 2025))
)
print(f"Dropping {(~mask_valid).sum()} rows that fail logical checks.")
df = df[mask_valid].reset_index(drop=True)
df.shape

### 3.8 Outliers — legit vs. errors

Look at the price distribution:

In [ ]:
df['price'].describe()

In [ ]:
# Top 10 highest prices
df.nlargest(10, 'price')[['neighborhood', 'sqm', 'price', 'description']]

Some are genuinely luxury apartments in Brera or Centro — those are
**legitimate outliers** and should be kept. Others might be data-entry errors.

A naive "remove outliers with 3-sigma" or IQR rule would throw away both.
A better approach: use a **price-per-square-meter** sanity check.

In [ ]:
df['price_per_sqm'] = df['price'] / df['sqm']
df['price_per_sqm'].describe()

In [ ]:
# Outliers in either direction on price/mq?
import numpy as np
q1, q3 = df['price_per_sqm'].quantile([0.01, 0.99])
print(f"99% of listings have prezzo/mq between {q1:.1f} and {q3:.1f}")
df[(df['price_per_sqm'] < q1) | (df['price_per_sqm'] > q3)][
    ['neighborhood', 'sqm', 'price', 'price_per_sqm']].head(10)

Now we can see the suspicious cases more clearly. The rest of the
"outliers in absolute price" are simply **expensive but reasonable** for
their size — luxury apartments. We **don't drop them**.

---
## 4. Aggregation: groupby, value_counts, pivot

Once data is clean, summary becomes meaningful.

### 4.1 `groupby`

The most-used pandas idiom. Pattern: **split-apply-combine**.

In [ ]:
# Average price by zone
df.groupby('neighborhood')['price'].mean().sort_values(ascending=False).head(10)

In [ ]:
# Multiple aggregations at once
df.groupby('neighborhood')['price'].agg(['mean', 'median', 'count']).sort_values('mean', ascending=False).head(10)

In [ ]:
# Group by two keys
df.groupby(['neighborhood', 'contract_type'])['price'].mean().head(15)

### 4.2 `value_counts`

Quick way to see the distribution of a categorical variable.

In [ ]:
df['contract_type'].value_counts()

In [ ]:
# As percentages
df['contract_type'].value_counts(normalize=True) * 100

### 4.3 `pivot_table`

In [ ]:
# Average price by zone × contract type
pivot = df.pivot_table(
    values='price',
    index='neighborhood',
    columns='contract_type',
    aggfunc='mean'
)
pivot.head(10)

`pivot_table` is just a more flexible `groupby` for the case where you
want a 2D table. The same result can always be achieved with `groupby` +
`unstack`, but `pivot_table` is more readable.

---
## 5. Merging with another table

We have a second file, `zone_milan.csv`, with extra info per zone:
distance to the Duomo, average income, transport score.

In [ ]:
zones = pd.read_csv('zones_milan.csv')
zones

Now merge it with our main DataFrame:

In [ ]:
before = len(df)
df_merged = df.merge(zones, on='neighborhood', how='left')
print(f"Before: {before} rows. After: {len(df_merged)} rows.")
print(f"Rows that didn't match: {df_merged['distance_duomo_km'].isna().sum()}")

**Always print before/after, and check for NaN on the right-hand columns.**
Some rows didn't match — let's see why.

In [ ]:
# Find unmatched zones
unmatched = df_merged[df_merged['distance_duomo_km'].isna()]['neighborhood'].unique()
unmatched

In [ ]:
# Compare: zones in zones file vs zones in main df
print("In main df, not in zones file:")
print(set(df['neighborhood'].unique()) - set(zones['neighborhood'].unique()))
print("\nIn zones file, not in main df:")
print(set(zones['neighborhood'].unique()) - set(df['neighborhood'].unique()))

The two files have the *same* zones, but with slightly different
spelling: `'Centro Storico'` vs `'Centro storico'`, `'NoLo'` vs `'Nolo'`,
`'De Angeli'` vs `'De angeli'`.

> **This is one of the most common bugs in real data engineering.** A merge
> that "works" can silently drop rows. Without the before/after check, you'd
> never notice.

Fix: normalize on both sides before merging.

In [ ]:
# Normalize both sides
df['zona_key'] = df['neighborhood'].str.lower()
zones['zona_key'] = zones['neighborhood'].str.lower()

df_merged = df.merge(zones.drop(columns='neighborhood'), on='zona_key', how='left')
print(f"Unmatched after fix: {df_merged['distance_duomo_km'].isna().sum()}")

> **🛑 LLM checkpoint.** Ask your LLM: *"Merge these two DataFrames on
> the `neighborhood` column."* See if it warns you about possible mismatches in
> spelling, or if it just gives you `df.merge(zones, on='neighborhood')` without
> a sanity check.

---
## 6. Visualization with matplotlib (and a taste of seaborn)

We end with the part that gives data its narrative power: **plotting**.

`matplotlib` is the foundational library. `seaborn` is a higher-level layer
on top of it, with prettier defaults and statistical shortcuts. `pandas`
itself has `.plot()` methods that wrap matplotlib.

Three things worth knowing up front:

1. There are **two APIs in matplotlib**: the "stateful" `plt.something()` and
   the "object-oriented" `fig, ax = plt.subplots()`. The OO API is
   recommended for anything beyond a quick plot. We'll use it.
2. A **figure** can contain multiple **axes** (subplots). Don't confuse the
   two.
3. Almost every plot starts with `fig, ax = plt.subplots(...)` and ends with
   labels, title, and `plt.show()`.

### 6.1 Histogram — the distribution of price

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['price'], bins=50, edgecolor='white')
ax.set_xlabel('Monthly rent (€)')
ax.set_ylabel('Number of listings')
ax.set_title('Distribution of monthly rents in Milan')
plt.show()

The distribution is right-skewed (a few very expensive listings pulling
the tail). When this happens, a log scale on the x-axis often helps:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['price'], bins=50, edgecolor='white')
ax.set_xscale('log')
ax.set_xlabel('Monthly rent (€, log scale)')
ax.set_ylabel('Number of listings')
ax.set_title('Distribution of monthly rents (log scale)')
plt.show()

### 6.2 Scatter plot — price vs square meters

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(df['sqm'], df['price'], alpha=0.3, s=10)
ax.set_xlabel('Surface (m²)')
ax.set_ylabel('Monthly rent (€)')
ax.set_title('Rent vs surface')
plt.show()

**`alpha=0.3`** makes points semi-transparent — essential for scatter
plots with many points, otherwise overlapping points look the same as a
single point. Always use it when you have more than a few hundred points.

### 6.3 Bar plot — average price by zone

In [ ]:
avg_by_neighborhood = df.groupby('neighborhood')['price'].mean().sort_values()

fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(avg_by_neighborhood.index, avg_by_neighborhood.values)
ax.set_xlabel('Average monthly rent (€)')
ax.set_title('Average rent by zone')
plt.show()

Horizontal bars (`barh`) are usually more readable than vertical bars
when the categories are many or have long names — vertical labels rotate and
become hard to read.

### 6.4 Boxplot — distribution by group

Boxplots are perfect for comparing distributions side by side.

In [ ]:
# Pick a few zones for clarity
top_zones = df['neighborhood'].value_counts().head(8).index
df_top = df[df['neighborhood'].isin(top_zones)]

fig, ax = plt.subplots(figsize=(10, 5))
df_top.boxplot(column='price', by='neighborhood', ax=ax, rot=45)
ax.set_ylabel('Monthly rent (€)')
ax.set_title('Rent distribution by zone (top 8 zones by count)')
plt.suptitle('')  # remove the automatic super-title
plt.tight_layout()
plt.show()

### 6.5 Multiple subplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['sqm'], bins=40, edgecolor='white')
axes[0].set_xlabel('Surface (m²)')
axes[0].set_title('Surface distribution')

axes[1].hist(df['year_built'].dropna(), bins=40, edgecolor='white')
axes[1].set_xlabel('Construction year')
axes[1].set_title('Construction year distribution')

plt.tight_layout()
plt.show()

### 6.6 A taste of seaborn

Seaborn shines for **statistical** plots. A few examples that are awkward
in plain matplotlib but trivial in seaborn:

In [ ]:
# Distribution of price by zone (only top zones), with ordered categories
fig, ax = plt.subplots(figsize=(10, 5))
order = df_top.groupby('neighborhood')['price'].median().sort_values().index
sns.boxplot(data=df_top, x='neighborhood', y='price', ax=ax, order=order)
plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('Rent by zone (sorted by median)')
plt.tight_layout()
plt.show()

In [ ]:
# Pairwise relationships among numeric columns
sns.pairplot(df[['price', 'sqm', 'rooms', 'year_built']].sample(500),
             diag_kind='hist', plot_kws={'alpha': 0.3, 's': 10})
plt.show()

In [ ]:
# Correlation heatmap
fig, ax = plt.subplots(figsize=(7, 5))
corr = df[['price', 'sqm', 'rooms', 'bedrooms', 'bathrooms',
           'year_built', 'condo_fees']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Correlations')
plt.tight_layout()
plt.show()

### 6.7 A word on plot quality

The plots above are *functional*. They are not *publication-quality*. Going
from one to the other (axis labels with units, font sizes, color choices that
are colorblind-friendly, removed gridlines, annotation of important points)
is a separate skill — and one where LLMs are genuinely useful, because the
syntax is fiddly.

> **🛑 Final LLM checkpoint.** Pick one of the plots above and ask your LLM:
> *"Improve this plot for inclusion in a research paper."* See what it
> suggests. Most of the time the suggestions are good (clear axis labels,
> bigger fonts, clean style). The remaining 10% — that's where you have to
> notice and intervene.

---
## Wrap-up

Today we did **not** learn pandas as a reference manual. We learned pandas as
a sequence of tools to answer questions about a real dataset:

| Step | Tools |
|---|---|
| Load and inspect | `read_csv`, `head`, `shape`, `dtypes`, `info`, `describe`, `isna().sum()` |
| Select and filter | `[]`, `loc`, `iloc`, boolean indexing, `isin` |
| Clean | `astype`, `str.replace`, `apply`, `map`, `to_datetime`, `fillna`, `drop_duplicates` |
| Aggregate | `groupby`, `agg`, `value_counts`, `pivot_table` |
| Merge | `merge` (with sanity checks!) |
| Visualize | `matplotlib`, `seaborn`, `df.plot` |

**The common thread:** for every operation, ask yourself *"how do I check that
it did what I expected?"* — by counting rows before and after, by inspecting
unique values, by plotting distributions. This is the discipline that
separates "code that runs" from "results you can trust".

This is also exactly the discipline you need to **work productively with
LLMs**: their outputs always run, but they do not always do what you intended.
Verification is your job.

### What's next

Next lecture: from descriptive analysis to **modeling** — supervised learning
on this same dataset, predicting the price from the other features. You'll
see why every cleaning shortcut we took matters.